# Single-Agent Smart Assistant + Quiz Auto-Grader

**Part 1** — single-agent pipeline: conditional routing, calculator tool, keyword-extraction tool, error handling.

**Part 2** — uses the keyword tool built in Part 1 to grade the Week 8 quiz answers against a reference keyword set and produce a score.

## Part 1 — Agent Pipeline

In [9]:
import re
import json
import time
import logging
from typing import Dict, List, Any

logging.basicConfig(level=logging.INFO, format='%(asctime)s - [%(levelname)s] - %(message)s')
agent_logger = logging.getLogger('agent_logger')


In [10]:
class MathEvaluatorTool:
    """Parses and evaluates mathematical expressions from queries."""
    name = "math_evaluator"

    def run(self, user_query: str) -> Dict[str, Any]:
        matched_expr = re.search(r"[\d\.\s\+\-\*\/\(\)]+", user_query)
        if not matched_expr:
            raise ValueError("No mathematical expression detected")
        expression = matched_expr.group().strip()

        if not re.fullmatch(r"[\d\.\s\+\-\*\/\(\)]+", expression):
            raise ValueError("Expression contains unsafe characters")
        
        calc_result = eval(expression, {"__builtins__": {}})
        return {"tool_name": self.name, "parsed_expression": expression, "result": calc_result}

IGNORED_WORDS = {
    "the","a","an","is","are","was","were","in","on","of","to","and","or","for",
    "with","that","this","it","as","by","be","from","at","which","how","what",
    "why","when","can","could","would","should","if","then","than","its","their",
    "these","those","also","such","into","over","between","each","other","more",
    "so","do","does","did","not","no","but","has","have","had","will","may",
    "used","use","using","provide","give","example","explain","describe","define",
}

class SalientKeywordTool:
    """Identifies and extracts key terms from provided text."""
    name = "keyword_extractor"

    def run(self, user_query: str) -> Dict[str, Any]:
        cleaned_text = re.sub(r"[^a-z0-9\s\-]", " ", user_query.lower())
        tokenized_words = [word.strip("-") for word in cleaned_text.split()]
        filtered_words = [word for word in tokenized_words if word and word not in IGNORED_WORDS and len(word) > 2]

        word_frequencies = {}
        for word in filtered_words:
            word_frequencies[word] = word_frequencies.get(word, 0) + 1
            
        sorted_keywords = sorted(set(filtered_words), key=lambda w: -word_frequencies[w])
        return {"tool_name": self.name, "extracted_keywords": sorted_keywords, "keyword_count": len(sorted_keywords)}


In [11]:
class SmartAgent:
    def __init__(self):
        self.math_tool = MathEvaluatorTool()
        self.keyword_tool = SalientKeywordTool()
        self.retry_limit = 3

    def determine_route(self, user_query: str) -> str:
        lowercase_query = user_query.lower()
        if "calculate" in lowercase_query or re.search(r"\d+\s*[\+\-\*\/]\s*\d+", lowercase_query):
            return "process_math"
        if "keyword" in lowercase_query:
            return "process_keyword"
        return "process_general"

    def execute_with_retries(self, func, *args):
        latest_error = None
        for attempt_num in range(1, self.retry_limit + 1):
            try:
                return func(*args)
            except Exception as err:
                latest_error = err
                agent_logger.info(f"Attempt {attempt_num} encountered an error: {err}")
                time.sleep(0)
        return {"error_message": str(latest_error)}

    def process_math(self, user_query: str, agent_state: dict):
        computation_result = self.execute_with_retries(self.math_tool.run, user_query)
        agent_state["output"] = computation_result
        agent_state["execution_path"].append("process_math")
        return agent_state

    def process_keyword(self, user_query: str, agent_state: dict):
        extraction_result = self.execute_with_retries(self.keyword_tool.run, user_query)
        agent_state["output"] = extraction_result
        agent_state["execution_path"].append("process_keyword")
        return agent_state

    def process_general(self, user_query: str, agent_state: dict):
        agent_state["output"] = {"tool_name": "general_handler", "response_text": f"Direct response for: {user_query}"}
        agent_state["execution_path"].append("process_general")
        return agent_state

    def execute(self, user_query: str) -> dict:
        current_state = {"original_query": user_query, "execution_path": ["determine_route"], "output": None}
        target_node = self.determine_route(user_query)
        node_handler = getattr(self, target_node)
        current_state = node_handler(user_query, current_state)
        return current_state

main_agent = SmartAgent()


In [12]:
sample_queries = [
    "calculate 12 + 8 * 2", 
    "extract keywords from this sentence about agents", 
    "tell me about the weather"
]
for query in sample_queries:
    print(f"{query} -> {main_agent.execute(query)}")


calculate 12 + 8 * 2 -> {'original_query': 'calculate 12 + 8 * 2', 'execution_path': ['determine_route', 'process_math'], 'output': {'tool_name': 'math_evaluator', 'parsed_expression': '12 + 8 * 2', 'result': 28}}
extract keywords from this sentence about agents -> {'original_query': 'extract keywords from this sentence about agents', 'execution_path': ['determine_route', 'process_keyword'], 'output': {'tool_name': 'keyword_extractor', 'extracted_keywords': ['about', 'extract', 'sentence', 'agents', 'keywords'], 'keyword_count': 5}}
tell me about the weather -> {'original_query': 'tell me about the weather', 'execution_path': ['determine_route', 'process_general'], 'output': {'tool_name': 'general_handler', 'response_text': 'Direct response for: tell me about the weather'}}


## Part 2 — Quiz Auto-Grader


In [13]:
QUIZ = [
    {
        "id": 1,
        "question": "Explain the concept of a stateful directed graph in agent pipelines. How does it differ from a simple linear pipeline?",
        "student_answer": "A stateful directed graph is a workflow where each step can store and use information from previous steps. The workflow is made up of nodes and edges that define how data moves through the system. It can support branching, looping, and decision-making. This makes it more flexible and intelligent. In contrast, a linear pipeline follows a fixed sequence of steps from start to finish without remembering previous states or changing its path.",
        "reference_keywords": ["state", "nodes", "edges", "branching", "looping", "decision-making", "linear", "fixed sequence"],
    },
    {
        "id": 2,
        "question": "Describe the role of nodes and edges in an agent workflow. Give an example of each.",
        "student_answer": "Nodes are the individual tasks or actions performed in an agent workflow. They can represent operations such as processing a query, calling a tool, or generating a response. Edges are the connections between nodes that determine how information flows. For example, a Calculator Tool can be a node, while the path connecting query analysis to the calculator is an edge. Together, nodes and edges define the complete workflow.",
        "reference_keywords": ["nodes", "tasks", "actions", "edges", "connections", "information flow", "example"],
    },
    {
        "id": 3,
        "question": "What is conditional routing in an agent system? Design a simple rule-based routing logic for three different query types.",
        "student_answer": "Conditional routing is the process of directing a query to different tools or actions based on its intent. It helps the agent choose the most appropriate response method. For example, if a query contains the word calculate, it is sent to the Calculator Tool. If it contains keywords, it is sent to the Keyword Extraction Tool. All other queries can be handled by a General Response module.",
        "reference_keywords": ["conditional routing", "intent", "calculator tool", "keyword extraction tool", "general response", "rule-based"],
    },
    {
        "id": 4,
        "question": "Why are cycles (loops) important in agent pipelines? Provide a use case where a retry loop is necessary.",
        "student_answer": "Cycles or loops allow an agent to repeat a process until a desired result is achieved. They are useful when tasks may fail or require multiple attempts. For example, if an API request fails due to a temporary network issue, the agent can retry the request instead of immediately returning an error. This improves reliability and robustness. Without loops, the workflow would stop after a single failure.",
        "reference_keywords": ["cycles", "loops", "repeat", "retry", "api request", "failure", "reliability"],
    },
    {
        "id": 5,
        "question": "Explain how a single-agent system can simulate multi-agent behavior internally.",
        "student_answer": "A single-agent system can simulate multiple agents by dividing its work into different roles. It may first analyze the query, then decide which tool to use, and finally generate a response. Each role behaves like a separate agent even though they are all executed by one system. This approach reduces complexity while maintaining flexibility. As a result, a single agent can perform tasks similar to a multi-agent architecture.",
        "reference_keywords": ["roles", "divide", "analyze query", "decide tool", "generate response", "single system", "multi-agent"],
    },
    {
        "id": 6,
        "question": "What are JSON schema tools? How do they help in structuring tool inputs and outputs?",
        "student_answer": "JSON schema tools define a standard structure for data exchanged between agents and tools. They specify required fields, data types, and expected formats. This ensures that inputs are valid before processing begins. It also makes outputs consistent and easier to interpret. Using JSON schemas reduces errors and improves communication between different components of an agent system.",
        "reference_keywords": ["json schema", "standard structure", "required fields", "data types", "validation", "consistent output"],
    },
    {
        "id": 7,
        "question": "Compare sequential tool calls and parallel tool calls. When would you prefer one over the other?",
        "student_answer": "Sequential tool calls are executed one after another, where each step depends on the result of the previous step. Parallel tool calls execute multiple independent tasks at the same time. Sequential execution is preferred when tasks have dependencies. Parallel execution is useful when tasks are unrelated and can be completed simultaneously. Using parallel calls can significantly reduce response time and improve efficiency.",
        "reference_keywords": ["sequential", "parallel", "dependencies", "independent tasks", "response time", "efficiency"],
    },
    {
        "id": 8,
        "question": "How would you implement error handling in a tool-using agent? Provide at least two strategies.",
        "student_answer": "Error handling helps an agent continue operating even when problems occur. One strategy is using try-except blocks to catch exceptions and return meaningful error messages. Another strategy is implementing retry mechanisms that automatically repeat failed operations. Logging errors is also useful for debugging and monitoring. These techniques improve reliability and help maintain a better user experience.",
        "reference_keywords": ["try-except", "exceptions", "retry mechanism", "logging", "debugging", "reliability"],
    },
    {
        "id": 9,
        "question": "What is trajectory evaluation in agent systems? Why is it important beyond just checking final output?",
        "student_answer": "Trajectory evaluation examines the entire sequence of actions taken by an agent to solve a task. It focuses on the decisions, tool calls, and intermediate steps used during execution. This helps identify mistakes that may not be visible in the final answer. It is useful for debugging, optimization, and improving agent performance. By evaluating the full trajectory, developers gain a deeper understanding of agent behavior.",
        "reference_keywords": ["trajectory", "sequence of actions", "decisions", "tool calls", "intermediate steps", "debugging", "optimization"],
    },
    {
        "id": 10,
        "question": "Define task completion rate and cost metrics. How would you measure and optimize them in a real-world system?",
        "student_answer": "Task completion rate measures the percentage of tasks successfully completed by an agent. Cost metrics represent the resources consumed, such as API calls, processing time, or monetary expenses. These metrics can be measured by tracking agent performance over multiple tasks. To optimize them, developers can improve routing accuracy, reduce unnecessary tool calls, and use efficient algorithms.",
        "reference_keywords": ["completion rate", "cost metrics", "resources", "api calls", "processing time", "routing accuracy", "optimize"],
    },
]
print(f"Loaded {len(QUIZ)} questions")


Loaded 10 questions


In [14]:
def clean_and_normalize(text_input: str) -> str:
    text_input = text_input.lower()
    text_input = re.sub(r"[^a-z0-9\s\-]", " ", text_input)
    return re.sub(r"\s+", " ", text_input).strip()

def evaluate_student_answer(quiz_item: dict, student_response: str, keyword_extractor: SalientKeywordTool) -> dict:
    """Scores a student's answer based on reference keywords."""
    extracted_data = keyword_extractor.run(student_response)
    extracted_terms = extracted_data.get("extracted_keywords", [])
    normalized_response = clean_and_normalize(student_response)

    found_keywords = []
    missing_keywords = []
    
    for reference in quiz_item["reference_keywords"]:
        norm_ref = clean_and_normalize(reference)
        if norm_ref in normalized_response or all(part in normalized_response for part in norm_ref.split()):
            found_keywords.append(reference)
        else:
            missing_keywords.append(reference)

    calculated_score = len(found_keywords) / len(quiz_item["reference_keywords"])
    return {
        "question_id": quiz_item["id"],
        "top_extracted_keywords": extracted_terms[:10],
        "matched_terms": found_keywords,
        "missed_terms": missing_keywords,
        "final_score": round(calculated_score, 2),
    }


In [15]:
def administer_quiz(quiz_data: list, keyword_extractor: SalientKeywordTool) -> list:
    """Iterates through quiz questions, collects answers, and evaluates them."""
    evaluation_results = []
    cumulative_score = 0.0
    
    for q_item in quiz_data:
        q_id = q_item["id"]
        q_text = q_item["question"]
        print(f"\nQuestion {q_id}: {q_text}")
        
        user_input = input("Your answer: ")
        eval_result = evaluate_student_answer(q_item, user_input, keyword_extractor)
        evaluation_results.append(eval_result)
        
        cumulative_score += eval_result["final_score"]
        percentage_score = eval_result["final_score"] * 100
        matches = eval_result["matched_terms"]
        misses = eval_result["missed_terms"]
        print(f"Score: {percentage_score:.0f}% | Matched: {matches} | Missed: {misses}")

    average_score = cumulative_score / len(quiz_data)
    print(f"\nTotal Overall Score: {average_score*100:.1f}% ({cumulative_score:.2f} out of {len(quiz_data)})")
    return evaluation_results

quiz_results = administer_quiz(QUIZ, main_agent.keyword_tool)



Question 1: Explain the concept of a stateful directed graph in agent pipelines. How does it differ from a simple linear pipeline?
Score: 100% | Matched: ['state', 'nodes', 'edges', 'branching', 'looping', 'decision-making', 'linear', 'fixed sequence'] | Missed: []

Question 2: Describe the role of nodes and edges in an agent workflow. Give an example of each.
Score: 100% | Matched: ['nodes', 'tasks', 'actions', 'edges', 'connections', 'information flow', 'example'] | Missed: []

Question 3: What is conditional routing in an agent system? Design a simple rule-based routing logic for three different query types.
Score: 50% | Matched: ['conditional routing', 'intent', 'calculator tool'] | Missed: ['keyword extraction tool', 'general response', 'rule-based']

Question 4: Why are cycles (loops) important in agent pipelines? Provide a use case where a retry loop is necessary.
Score: 71% | Matched: ['cycles', 'loops', 'repeat', 'retry', 'api request'] | Missed: ['failure', 'reliability']

Q